# Visualize Output for EXP_04

Remark : Please use your IDE in Light mode for better UI :)

## Config & Setup

In [3]:
import os
import json
from IPython.display import display, HTML

# 1. ตั้งค่า Folder และ ชื่อไฟล์ที่ต้องการตรวจสอบ
FOLDER_PATH = 'dataset/output_14/'
FILE_NAME = 'รายงานสืบสวนอุบัติเหตุเชิงลึก_KU-250302-54_Option1_NoBG.json' # <-- เปลี่ยนชื่อไฟล์ตรงนี้

file_path = os.path.join(FOLDER_PATH, FILE_NAME)

# 2. โหลดข้อมูล
try:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"✅ โหลดข้อมูลสำเร็จ: {FILE_NAME}")
except FileNotFoundError:
    print(f"❌ ไม่พบไฟล์: {file_path}")
except json.JSONDecodeError:
    print("❌ ไม่สามารถอ่านไฟล์ JSON ได้ (Format อาจพัง)")

✅ โหลดข้อมูลสำเร็จ: รายงานสืบสวนอุบัติเหตุเชิงลึก_KU-250302-54_Option1_NoBG.json


## Render UI Function

In [4]:
def render_iteration_history(history_list, title):
    if not history_list:
        display(HTML(f"<h3 style='color: red;'>❌ ไม่มีข้อมูลประวัติสำหรับ {title}</h3>"))
        return

    display(HTML(f"<h2 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 5px;'>=== ประวัติการรัน: {title} ===</h2>"))
    
    for item in history_list:
        iteration = item.get('iteration', 'N/A')
        score = item.get('judge_score', 0)
        reasoning = item.get('judge_reasoning', 'N/A')
        
        # จัดสีตามคะแนน (ผ่านเกณฑ์ 8 ให้สีเขียว, ไม่ผ่านให้สีแดง)
        score_color = "#27ae60" if score >= 8 else "#e74c3c"
        
        # จัด Format ให้ LLM1 Output ดูง่ายขึ้น
        generated_raw = item.get('generated', '{}')
        try:
            # พยายามแกะ JSON string ให้ออกมาเป็น Dict จะได้แสดงผลสวยๆ
            gen_dict = json.loads(generated_raw)
            thought = gen_dict.get('thought_process', 'ไม่มีข้อมูล')
            
            # ดึงข้อมูลส่วนเนื้อหามาแสดง (ลบ thought ออกไปก่อนจะได้ไม่รก)
            content_dict = {k: v for k, v in gen_dict.items() if k != 'thought_process'}
            content_json = json.dumps(content_dict, ensure_ascii=False, indent=2)
            
            llm1_html = f"""
                <div style='margin-bottom: 10px;'><b>🧠 Thought Process:</b><br><span style='color: #555;'>{thought}</span></div>
                <div><b>📝 Generated Output:</b><br><pre style='background: #eee; padding: 5px; border-radius: 4px; font-size: 12px;'>{content_json}</pre></div>
            """
        except Exception as e:
            # ถ้า Format พัง (เช่น รอบที่ JSON พัง) ให้โชว์แบบดิบๆ
            llm1_html = f"<b>⚠️ Raw Output (JSON Error):</b><br><pre style='background: #f8d7da; padding: 5px; font-size: 12px; white-space: pre-wrap;'>{generated_raw}</pre>"

        # สร้าง HTML Box
        html_output = f"""
        <div style="border: 2px solid #bdc3c7; border-radius: 8px; margin-bottom: 20px; box-shadow: 2px 2px 5px rgba(0,0,0,0.1);">
            <div style="background-color: #ecf0f1; padding: 10px; border-top-left-radius: 6px; border-top-right-radius: 6px; display: flex; justify-content: space-between;">
                <b style="font-size: 16px;">🔄 Iteration: {iteration}</b>
                <b style="font-size: 16px; color: {score_color};">🎯 Judge Score: {score}/10</b>
            </div>
            
            <div style="display: flex; flex-direction: row; padding: 0;">
                <div style="flex: 1; padding: 15px; border-right: 1px solid #bdc3c7;">
                    <h4 style="color: #2980b9; margin-top: 0;">🤖 LLM1 (Generator)</h4>
                    {llm1_html}
                </div>
                
                <div style="flex: 1; padding: 15px; background-color: #fdfefe;">
                    <h4 style="color: #c0392b; margin-top: 0;">⚖️ LLM2 (Judge Feedback)</h4>
                    <p style="white-space: pre-wrap; font-family: monospace; font-size: 13px; line-height: 1.5; color: #333;">{reasoning}</p>
                </div>
            </div>
        </div>
        """
        display(HTML(html_output))

## UI: Factors

In [5]:
# แสดงผลลูปของการสร้าง Causes
display_causes = data.get('history_causes', [])
render_iteration_history(display_causes, "Factor/Causes Generation")

## UI: Solutions

In [6]:
# แสดงผลลูปของการสร้าง Solutions
display_solutions = data.get('history_solutions', [])
render_iteration_history(display_solutions, "Solutions Generation")

## Final Accepted Output

In [7]:
print("="*60)
print("🏆 สรุปผลลัพธ์สุดท้าย (Final Results)")
print("="*60)

print("\n[Metrics การรันลูป]")
for k, v in data.get('experiment_metrics', {}).items():
    print(f" - {k}: {v}")

print("\n[Final Causes]")
print(json.dumps(data.get('generated_causes', {}), ensure_ascii=False, indent=2))

print("\n" + "-"*40)

print("\n[Final Solutions]")
print(json.dumps(data.get('generated_solutions', {}), ensure_ascii=False, indent=2))

🏆 สรุปผลลัพธ์สุดท้าย (Final Results)

[Metrics การรันลูป]
 - use_factor_bg: False
 - use_solution_bg: False
 - cause_iterations_needed: 2
 - solution_iterations_needed: 1

[Final Causes]
{
  "thought_process": "รายงานระบุว่า 'ผู้ขับขี่รถยนต์ส่วนบุคคล Nissan Terra สีขาว ขับมาด้วยความเร็ว 90 กม./ชม.' และ 'ชนท้ายรถจักรยานยนต์ Honda Wave 125i' ซึ่งแสดงให้เห็นว่าผู้ขับขี่รถยนต์ขับรถด้วยความเร็วสูงเกินไป",
  "main_cause_group": "ยานพาหนะ",
  "cause_list": [
    {
      "group": "ยานพาหนะ",
      "items": [
        "ผู้ขับขี่รถยนต์ขับรถด้วยความเร็วสูงเกินไป"
      ]
    }
  ]
}

----------------------------------------

[Final Solutions]
{
  "thought_process": "รายงานระบุว่า 'ผู้ขับขี่รถยนต์ส่วนบุคคล Nissan Terra สีขาว ขับมาด้วยความเร็ว 90 กม./ชม.' และ 'ชนท้ายรถจักรยานยนต์ Honda Wave 125i' ซึ่งแสดงให้เห็นว่าผู้ขับขี่รถยนต์ขับรถด้วยความเร็วสูงเกินไป",
  "short_term_solutions": [
    "จัดทำป้ายเตือนความเร็วและไฟสัญญาณเตือนที่ทางตรงเพื่อให้ผู้ขับขี่ทราบถึงความเร็วที่เหมาะสม",
    "จัดการฝึกอบรมค